# LiciGuard · Historial de licitaciones (datos abiertos OCDS) automáticamente

Este cuaderno arma los datos para la pregunta **"¿Puedo ganar?"**. Hace todo solo:

```
1. Descarga los datos abiertos  →  2. Clasifica cada compra por rubro  →  3. Guarda tus rubros  →  4. Valida
   del OECE (año por año)            (con reglas y prioridades)            en historial.csv          la clasificación
```

Con `historial.csv` la app puede decir, por ejemplo: *"Textiles del Norte SAC ganó 4 de 4 procesos parecidos en esta entidad, casi siempre con un solo postor"*.

**Cómo usarlo:** ábrelo en Google Colab y ejecuta las celdas en orden con `Shift + Enter`.

## Paso 1 · Rubros y años

**Cómo se escriben las palabras** (todo en minúsculas y sin tildes):
- `"silla"` → busca la palabra completa: *silla, sillas* (no *sillar*).
- `"mueble*"` → con asterisco busca todo lo que empiece así: *mueble, muebles, mueblería*.
- Frases como `"papel higienico"` son más precisas que palabras sueltas como `"papel"`.

**Prioridad:** si una compra coincide con varios rubros, gana la frase **más específica (más larga)**.
Ejemplo: *"servicio de limpieza"* → Servicios generales (y no Limpieza); *"papel higienico"* → Limpieza (y no Útiles de oficina).

In [ ]:
RUBROS = {
    "Uniformes y textiles": ["uniforme", "prenda* de vestir", "vestuario", "confeccion*", "textil*", "chaleco", "polo",
                             "camisa", "pantalon*", "casaca", "chompa", "mandil*", "buzo", "gorro", "tela", "calzado",
                             "zapato", "botin*", "bota"],
    "Mobiliario":           ["mobiliario", "mueble*", "carpeta* escolar*", "carpeta* unipersonal*", "carpeta* bipersonal*",
                             "escritorio", "silla", "estante*", "estanteria*", "archivador* metalico*", "pizarra",
                             "mesa", "armario", "locker"],
    "Útiles de oficina":    ["utiles de oficina", "utiles de escritorio", "material de escritorio", "material* de oficina",
                             "papel bond", "papel a4", "toner*", "toner* para impresora*", "cartucho* de tinta", "cartucho* para impresora*", "tinta* para impresora*", "lapicero*", "folder*",
                             "archivador* de palanca", "resaltador*", "engrapador*", "corrector*", "sobre* manila"],
    "Limpieza":             ["limpieza", "material* de limpieza", "articulo* de limpieza", "producto* de limpieza",
                             "detergente*", "desinfectante*", "lejia", "papel higienico", "papel toalla", "jabon*",
                             "escoba*", "trapeador*", "bolsa* de basura", "bolsa* para residuos", "alcohol en gel"],
    "Alimentos":            ["alimento*", "viveres", "racion*", "desayuno*", "arroz", "leche", "menestra*", "azucar",
                             "aceite vegetal", "conserva*", "harina", "fideo*", "pan", "canasta* basica*"],
    "Equipos de cómputo":   ["computadora*", "laptop*", "impresora*", "monitor*", "equipo* de computo",
                             "equipo* informatico*", "servidor* de datos", "tablet*", "proyector* multimedia", "escaner*"],
    "Telecomunicaciones":   ["router*", "switch* de red", "cable* de red", "cable* utp", "fibra optica", "access point",
                             "punto* de acceso", "modem*", "central telefonica", "antena*", "radio* de comunicacion*"],
    "Ferretería":           ["ferreteria", "herramienta*", "pintura*", "cemento", "tuberia*", "cable* electrico*",
                             "clavo*", "candado*", "cerradura*"],
    "Materiales de construcción": ["material* de construccion", "agregado*", "ladrillo*", "fierro", "acero corrugado",
                             "arena gruesa", "piedra chancada"],
    "Medicamentos e insumos médicos": ["medicamento*", "farmaco*", "producto* farmaceutico*", "insumo* medico*",
                             "material medico", "guante* quirurgico*", "jeringa*", "mascarilla*", "reactivo*",
                             "dispositivo* medico*", "oxigeno medicinal"],
    "Equipos médicos":      ["equipo* medico*", "equipamiento hospitalario", "ecografo*", "monitor* de funciones vitales",
                             "desfibrilador*", "camilla*", "autoclave*"],
    "Combustibles":         ["combustible*", "gasolina", "diesel", "petroleo", "gasohol", "glp", "gnv", "lubricante*"],
    "Vehículos y repuestos": ["vehiculo*", "camioneta*", "motocicleta*", "ambulancia*", "repuesto*", "neumatico*", "llanta*"],
    "Seguridad y EPP":      ["equipo* de proteccion personal", "epp", "casco* de seguridad", "guante* de seguridad",
                             "bota* de seguridad", "extintor*", "senalizacion*"],
    "Material educativo":   ["material educativo", "material didactico", "material pedagogico", "texto* escolar*",
                             "libro*", "cuaderno* de trabajo", "juego* didactico*"],
    "Impresión y publicaciones": ["servicio* de impresion", "impresion de*", "gigantografia*", "encuadernacion",
                             "empastado", "banner*", "afiche*", "folleto*", "triptico*", "volante"],
    "Servicios generales":  ["servicio* de limpieza", "servicio* de mantenimiento", "mantenimiento preventivo",
                             "mantenimiento correctivo", "servicio* de vigilancia", "servicio* de seguridad",
                             "servicio* de transporte", "servicio* de mensajeria", "servicio* de fumigacion",
                             "servicio* de lavanderia", "servicio* de alimentacion", "servicio* de reparacion"],
    "Consultorías":         ["consultoria*", "asesoria*", "expediente tecnico", "elaboracion de expediente*",
                             "estudio* de preinversion", "supervision de obra*"],
    "Obras":                ["ejecucion de obra*", "ejecucion de la obra*", "construccion de*", "mejoramiento de*",
                             "rehabilitacion de*", "ampliacion de*"],
}

# Elige los rubros de las bases que vas a mostrar. "TODOS" guarda todos los rubros (archivo grande).
ELEGIDOS = ["Uniformes y textiles", "Mobiliario", "Limpieza", "Útiles de oficina"]

AÑOS = ["2021", "2022", "2023", "2024", "2025", "2026"]
BORRAR_DESPUES = True  # borra cada archivo grande después de filtrarlo (ahorra espacio)
ENTIDADES = []         # opcional, por ejemplo ["municipalidad de lima", "essalud"]; vacío = todas
LIMITE = 100_000       # prueba rápida. None = leer todo

In [ ]:
# Paso 2 · Clasificador (normaliza el texto y aplica las prioridades)
import re, unicodedata

def normalizar(texto):
    """minúsculas, sin tildes, sin signos, espacios uniformes"""
    t = unicodedata.normalize("NFD", str(texto or "").lower())
    t = "".join(ch for ch in t if unicodedata.category(ch) != "Mn")
    return re.sub(r"[^a-z0-9*]+", " ", t).strip()

def patron(entrada):
    partes = [re.escape(p[:-1]) + r"[a-z]*" if p.endswith("*") else re.escape(p) + r"(?:s|es)?"
              for p in normalizar(entrada).split()]
    return re.compile(r"\b" + r"\s+".join(partes) + r"\b")

REGLAS = [(rubro, entrada, patron(entrada)) for rubro, lista in RUBROS.items() for entrada in lista]

def clasificar(descripcion):
    """Devuelve (rubro, palabra que decidió, rubros posibles). Gana la coincidencia más específica."""
    t = normalizar(descripcion).replace("*", " ")
    if not t:
        return "Sin descripción", "", []
    hallados = []
    for rubro, entrada, rx in REGLAS:
        m = rx.search(t)
        if m:
            hallados.append((len(m.group(0)), rubro, m.group(0)))
    if not hallados:
        return "Sin clasificar", "", []
    hallados.sort(key=lambda x: x[0], reverse=True)
    posibles = list(dict.fromkeys(r for _, r, _ in hallados))
    return hallados[0][1], hallados[0][2], posibles

def tipo(descripcion, rubro):
    t = normalizar(descripcion)
    if rubro == "Obras" or re.search(r"\bejecucion de (la )?obra", t):
        return "Obra"
    if rubro in ("Servicios generales", "Consultorías") or re.search(r"\bservicio", t):
        return "Servicio"
    return "Bien"

# Prueba rápida con descripciones de ejemplo
for d in ["ADQUISICIÓN DE UNIFORMES PARA EL PERSONAL", "COMPRA DE PAPEL HIGIÉNICO", "SUMINISTRO DE TÓNER PARA IMPRESORA",
          "SERVICIO DE LIMPIEZA DE LOCALES", "ADQUISICIÓN DE BOTAS DE SEGURIDAD", "COMPRA DE CARPETAS ESCOLARES",
          "MANTENIMIENTO PREVENTIVO DE EQUIPOS", "COMPRA DE ARROZ Y MENESTRAS", "ADQUISICIÓN DE SILLARES"]:
    r, palabra, posibles = clasificar(d)
    print(f"{d:42} → {r:22} (por «{palabra}»)" + (f"  · también: {posibles[1:]}" if len(posibles) > 1 else ""))

In [ ]:
# Paso 3 · Preparar la descarga
import os, requests

os.makedirs("ocds", exist_ok=True)
BASE = "https://data.open-contracting.org/en/publication/135/download?name={}.jsonl.gz"

def descargar(url, destino):
    with requests.get(url, stream=True, timeout=120, headers={"User-Agent": "Mozilla/5.0 (LiciGuard)"}) as r:
        r.raise_for_status()
        total, hecho, aviso = int(r.headers.get("Content-Length", 0)), 0, 0
        with open(destino + ".parcial", "wb") as f:
            for trozo in r.iter_content(chunk_size=1 << 20):
                f.write(trozo)
                hecho += len(trozo)
                if hecho - aviso > 100e6:
                    aviso = hecho
                    print(f"   {hecho/1e6:,.0f} MB" + (f" de {total/1e6:,.0f} MB" if total else ""))
    os.rename(destino + ".parcial", destino)

elegidos = list(RUBROS) if ELEGIDOS == "TODOS" else ELEGIDOS
# Descarte rápido sobre el texto crudo: la palabra más larga de cada regla elegida
claves = sorted({max(normalizar(e).replace("*", "").split(), key=len) for r in elegidos for e in RUBROS[r]})
FILTRO = re.compile("|".join(map(re.escape, claves)))
ESCAPES = {"\\u00e1": "a", "\\u00e9": "e", "\\u00ed": "i", "\\u00f3": "o", "\\u00fa": "u", "\\u00f1": "n",
           "\\u00c1": "a", "\\u00c9": "e", "\\u00cd": "i", "\\u00d3": "o", "\\u00da": "u", "\\u00d1": "n", "\\u00fc": "u"}
def crudo_normalizado(linea):
    for k, v in ESCAPES.items():
        linea = linea.replace(k, v)
    return normalizar(linea)
print("Rubros elegidos:", elegidos)

In [ ]:
# Paso 4 · Descargar, clasificar y crear historial.csv, año por año (puede tardar: deja la pestaña abierta)
import csv, gzip, json

def procesos(linea):
    if "compiledRelease" in linea: yield linea["compiledRelease"]
    elif isinstance(linea.get("releases"), list): yield from linea["releases"]
    else: yield linea

def fila(rel):
    tender, awards = rel.get("tender") or {}, rel.get("awards") or []
    activos = [a for a in awards if (a.get("status") or "active") == "active"] or awards
    ganadores = [s.get("name") for a in activos for s in (a.get("suppliers") or []) if s.get("name")]
    n_post = tender.get("numberOfTenderers")
    if n_post is None and tender.get("tenderers") is not None:
        n_post = len(tender["tenderers"])
    if n_post is None:
        n_post = sum("tenderer" in (p.get("roles") or []) for p in rel.get("parties") or []) or None
    docs = tender.get("documents") or []
    return {
        "ocid": rel.get("ocid"),
        "fecha": ((tender.get("tenderPeriod") or {}).get("startDate") or rel.get("date") or "")[:10],
        "entidad": (rel.get("buyer") or {}).get("name"),
        "titulo": " ".join(x for x in [tender.get("title"), tender.get("description")] if x)[:300],
        "valor_estimado": (tender.get("value") or {}).get("amount"),
        "n_postores": n_post,
        "ganador": " / ".join(dict.fromkeys(ganadores)) or None,
        "monto_adjudicado": sum(((a.get("value") or {}).get("amount") or 0) for a in activos) or None,
        "url": docs[0].get("url") if docs else None,
    }

COLUMNAS = ["rubro", "tipo", "palabra_clave", "rubros_posibles", "ocid", "fecha", "entidad", "titulo",
            "valor_estimado", "n_postores", "ganador", "monto_adjudicado", "url"]
entidades = [normalizar(e) for e in ENTIDADES]
guardados = 0
with open("historial.csv", "w", newline="", encoding="utf-8") as salida:
    w = csv.DictWriter(salida, fieldnames=COLUMNAS); w.writeheader()
    for año in AÑOS:
        archivo = f"ocds/{año}.jsonl.gz"
        if not os.path.exists(archivo):
            print(f"\n⬇️  Descargando {año}…")
            try:
                descargar(BASE.format(año), archivo)
            except Exception as e:
                print("   ❌ No se pudo:", e); continue
        print(f"🔎 Clasificando {año} ({os.path.getsize(archivo)/1e6:,.0f} MB)…")
        antes = guardados
        with gzip.open(archivo, "rt", encoding="utf-8") as entrada:
            for n, linea in enumerate(entrada, 1):
                if LIMITE and n > LIMITE: break
                if n % 200_000 == 0: print(f"   {año}: {n:,} líneas · {guardados:,} guardados")
                if not FILTRO.search(crudo_normalizado(linea)):      # descarte rápido
                    continue
                try:
                    obj = json.loads(linea)
                except json.JSONDecodeError:
                    continue
                for rel in procesos(obj):
                    f = fila(rel)
                    if not f["ganador"]:
                        continue
                    rubro, palabra, posibles = clasificar(f["titulo"])
                    if rubro not in elegidos:
                        continue
                    if entidades and not any(e in normalizar(f["entidad"]) for e in entidades):
                        continue
                    f.update(rubro=rubro, tipo=tipo(f["titulo"], rubro), palabra_clave=palabra,
                             rubros_posibles=" | ".join(posibles))
                    w.writerow(f); guardados += 1
        print(f"   ✅ {año}: {guardados - antes:,} procesos")
        salida.flush()
        if BORRAR_DESPUES and not LIMITE:
            os.remove(archivo)
print(f"\n✅ historial.csv creado con {guardados:,} procesos")

In [ ]:
# Paso 5 · Mira lo que encontraste
import pandas as pd
h = pd.read_csv("historial.csv")
print(len(h), "procesos\n")
if len(h):
    print("📦 Por rubro y tipo:"); print(pd.crosstab(h["rubro"], h["tipo"]).to_string(), "\n")
    print("🏛️  Entidades que más compran:"); print(h["entidad"].value_counts().head(8).to_string(), "\n")
    print("🏆 Proveedores que más ganan:"); print(h["ganador"].value_counts().head(8).to_string(), "\n")
    if h["n_postores"].notna().any():
        print(f"👥 Postores en promedio: {h['n_postores'].mean():.1f}  ·  con un solo postor: {(h['n_postores'] == 1).mean():.0%}")
h.head(10)

## Paso 6 · Validar la clasificación (mejóralo con tus datos reales)

Revisa estas 3 cosas y, si ves errores, ajusta `RUBROS` en el Paso 1 y vuelve a ejecutar:
1. **Casos ambiguos:** compras que coincidieron con varios rubros. ¿Ganó el correcto?
2. **Palabras que decidieron:** si una palabra clasifica mal muchas compras, hazla más específica.
3. **Palabras frecuentes por rubro:** te sugieren términos nuevos para agregar.

In [ ]:
from collections import Counter
VACIAS = set("de la el los las para del y en con por a al adquisicion compra suministro contratacion servicio bienes "
             "segun especificaciones tecnicas destinado destinados uso diversas diversos item items lote".split())
if len(h):
    amb = h[h["rubros_posibles"].fillna("").str.contains("|", regex=False)]
    print(f"1) Casos ambiguos: {len(amb)} de {len(h)} ({len(amb)/len(h):.0%})")
    print(amb[["titulo", "rubro", "palabra_clave", "rubros_posibles"]].head(15).to_string())
    print("\n2) Palabras que más decidieron, por rubro:")
    print(h.groupby("rubro")["palabra_clave"].agg(lambda s: ", ".join(f"{p} ({n})" for p, n in Counter(s).most_common(5))).to_string())
    print("\n3) Palabras frecuentes en los títulos, por rubro (ideas para ampliar el diccionario):")
    for rubro, grupo in h.groupby("rubro"):
        palabras = Counter(w for t in grupo["titulo"] for w in normalizar(t).split() if len(w) > 3 and w not in VACIAS)
        print(f"   {rubro}: {', '.join(p for p, _ in palabras.most_common(15))}")

In [ ]:
# Paso 7 · Descargar historial.csv
try:
    from google.colab import files
    files.download("historial.csv")
    print("Descargando… guárdalo en la carpeta data/ocds/ de tu proyecto.")
except ImportError:
    print("Copia historial.csv a la carpeta data/ocds/ de tu proyecto.")

## Listo 🎉

1. Pon `historial.csv` en `data/ocds/` de tu proyecto y reinicia la app.
2. En la barra lateral verás **"Datos abiertos: N procesos"**.
3. **"¿Puedo ganar?"** mostrará quién ganó antes en la misma entidad, con cuántos postores y a qué precio.

**Consejos**
- Primero prueba con `LIMITE = 100_000`. Cuando estés conforme con la validación, pon `LIMITE = None`.
- Si Colab se desconecta, vuelve a ejecutar poniendo en `AÑOS` solo los años que faltaban.
- Si la descarga falla: baja el `.jsonl.gz` desde https://data.open-contracting.org/en/publication/135 y súbelo a la carpeta `ocds` de Colab (panel de archivos a la izquierda); luego ejecuta desde el Paso 4.